# Labeling the Dataset using the trained student

What we do:
1) Run the student model on the dataset of inputs
2) Analyse the student dataset and label it


Student: Qwen2.5-1.5B-Instruct

In [1]:
import dotenv

dotenv.load_dotenv()

True

In [2]:
from core.types import *
from core.utils.huggingface_client import HuggingFaceClient
from core.utils.huggingface_inference_client import HuggingFaceInferenceClient
from core.utils.ollama_inference_client import OllamaInferenceClient
from core.utils.openai_client import OpenAIClient, ProcessingMode
from doom.preprocessing.doom_game_state_perturbator import DoomGameStatePerturbator
from doom.utils.doom_game_state import DoomGameState, MonsterType, WeaponName, AimedAtType
from sklearn.cluster import DBSCAN
from dataclasses import dataclass, asdict
from collections import Counter
from typing import Iterable
from pathlib import Path
from ollama import ChatResponse
from openai.types.responses import Response as OpenAIResponse

import os
import json
import numpy as np
import pandas as pd

/home/filippo/gamepals-llm-distillation/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
inputs = LLMCommandingInput.load_inputs(
    path=Path("data/inputs/inputs.json"),
    gstype=DoomGameState
)

inputs_lookup = {inp.id: inp for inp in inputs}

In [5]:
# For now, only extract inputs to label
selected_inputs = [
    inp
    for inp in inputs
    if inp.selected_for_labelling
]

print(f"Selected inputs: {len(selected_inputs)}/{len(inputs)}")

Selected inputs: 50/2872


In [12]:
student_client = HuggingFaceInferenceClient[LLMCommandingInput, LLMCommandingOutput](
    #model = "Qwen/Qwen2.5-1.5B-Instruct",
    model = "./data/huggingface/training/final",
    max_output_tokens=512,
    temperature=0.0,
    working_dir=Path('data/huggingface'),
    use_flash_attention_2=False,
    load_in_4bit=True
)

🚀 Initialized HuggingFaceInferenceClient for ./data/huggingface/training/final
   Device: cuda
   Flash Attention 2: False
   Quantization: 4-bit


In [13]:
# Prepare Prompt (same as training)
system_prompt = "You are a game command parser that converts natural language commands into DSL instructions."

In [14]:
def format_input(inp: LLMCommandingInput) -> str:
    game_state = inp.game_state.state.to_prompt_ready()
    command = inp.user_command.command.command
    return f"Game State:\n{game_state}\nCommand:\n{command}"


def parse_output(response: str, input_id: str, latency: float) -> LLMCommandingOutput:
    return LLMCommandingOutput(
        input_id=input_id,
        actions=response,
        reason=None,
        latency=latency,
    )


def get_id(gse: LLMCommandingInput, idx: int) -> str:
    return gse.id

In [15]:
print(system_prompt)
print(format_input(inputs[3]))

You are a game command parser that converts natural language commands into DSL instructions.
Game State:
AIMED_AT:
  type: Wall
  distance: 330.86
  interactable: yes

MONSTERS (count=0):

INVENTORY:
  current_slot: 2
  weapons:
    - (1, Fist, 0)
    - (2, Pistol, 50)
Command:
Go press that switch ahead


In [16]:
outputs = student_client.process(
    dataset=selected_inputs, # TO CHANGE
    system_prompt=system_prompt,
    tools = [], # No tools at level 3
    format_input=format_input,
    parse_output=parse_output,
    get_id=get_id,
    # batch_size=200,
)


📦 Loading model: ./data/huggingface/training/final
   📉 Using 4-bit quantization (NF4)


Loading weights: 100%|████████████████████████| 338/338 [00:00<00:00, 488.68it/s, Materializing param=model.norm.weight]


AttributeError: `weight` is not an nn.Module